<div class="hero" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <h1 style="direction:rtl;text-align:right;">داده بورس از tsetmc</h1>
  <p>
    هفته ۴ · بخش دوم · Zero to Quant — ارشد مدیریت مالی<br>
    اول می‌بینی سایت مدیریت فناوری بورس تهران قیمت را از کدام آدرس می‌گیرد.
    بعد همان داده را با کتابخانه <b>pytse-client</b> می‌خوانی.
    این کتابخانه روی داده زنده تست شده است.
  </p>
  <div class="hero-meta">
    <span class="chip">tsetmc</span>
    <span class="chip">pytse-client</span>
    <span class="chip">لیست و دیکشنری</span>
    <span class="chip">۱۶ کیس</span>
  </div>
</div>


<div class="guide" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <h2 style="direction:rtl;text-align:right;">۰) فونت و نصب</h2>
  <div class="step"><b>مرحله ۱:</b> سلول فونت را اجرا کن.</div>
  <div class="step"><b>مرحله ۲:</b> در ترمینال بنویس <span class="kbd">pip install pytse-client</span></div>
  <div class="step"><b>اگر نصب روی خط lxml ایستاد</b> این دو دستور را به ترتیب بزن:
  <span class="kbd">pip install "lxml&gt;=5" jdatetime tenacity aiohttp pandas requests beautifulsoup4</span>
  و بعد
  <span class="kbd">pip install pytse-client --no-deps</span></div>
  <div class="step"><b>مرحله ۳:</b> سلول <span class="kbd">load</span> را اجرا کن. بعد در هر سؤال با <span class="kbd">load("فولاد")</span> یا اسم نماد دیگری داده را بیاور. سؤال‌ها پانداس نمی‌خواهند.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> قالب خروجی شکل خط را نشان می‌دهد. قیمت و حجم با بازار عوض می‌شوند.</div>
</div>


In [ ]:
from IPython.display import HTML, display
from pathlib import Path

style = Path("fonts/notebook_style.html").read_text(encoding="utf-8")
display(HTML(style))
print("استایل و فونت وزیر اعمال شد.")


<div class="guide" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <h2 style="direction:rtl;text-align:right;">۱) آدرس‌ها را از پنل Network دربیاور</h2>
  <ol>
    <li>برو به <span class="kbd">https://www.tsetmc.com</span></li>
    <li>ابزار توسعه‌دهنده را باز کن. در ویندوز و مک کلید F12 است. زبانه Network را انتخاب کن.</li>
    <li>فیلتر را روی Fetch/XHR بگذار و صفحه را رفرش کن.</li>
    <li>یک نماد را باز کن، مثلا جستجوی فولاد. آدرس صفحه این شکل است: <span class="kbd">/instInfo/46348559193224090</span>. عدد آخر، کد نماد است و insCode نام دارد.</li>
  </ol>
  <p>در فهرست درخواست‌ها این‌ها را پیدا می‌کنی. میزبان همه‌شان <span class="kbd">cdn.tsetmc.com</span> است.</p>
  <ul>
    <li><b>جستجوی نماد:</b> <span class="kbd">/api/Instrument/GetInstrumentSearch/فولاد</span></li>
    <li><b>تابلوی لحظه:</b> <span class="kbd">/api/ClosingPrice/GetClosingPriceInfo/کد</span> — آخرین قیمت فیلد pDrCotVal، پایانی pClosing، دیروز priceYesterday. واحد ریال است.</li>
    <li><b>تاریخچه روزانه:</b> <span class="kbd">/api/ClosingPrice/GetClosingPriceDailyList/کد/0</span></li>
    <li><b>فهرست کل بازار:</b> <span class="kbd">/api/ClosingPrice/GetMarketWatch</span> — نماد در lva، پایانی لحظه در pcl، آخرین در pdv.</li>
    <li><b>دفتر سفارش:</b> <span class="kbd">/api/BestLimits/کد</span></li>
    <li><b>حقیقی و حقوقی:</b> <span class="kbd">/api/ClientType/GetClientTypeHistory/کد/تاریخ</span></li>
    <li><b>سهامدار عمده:</b> <span class="kbd">/api/Shareholder/GetInstrumentShareHolderLast/کد</span></li>
    <li><b>شاخص کل:</b> <span class="kbd">/api/Index/GetIndexB1LastDay/32097828799138957</span></li>
  </ul>
  <p>سلول بعدی یکی از همین آدرس‌ها را با requests می‌خواند تا مسیر JSON را ببینی. سؤال‌های بعد اسم نماد را به تابع <code>load</code> می‌دهند تا لازم نباشد کد عددی را حفظ کنی.</p>
</div>


In [ ]:
import requests

# همان درخواستی که صفحه نماد در پنل Network می‌زند.
url = "https://cdn.tsetmc.com/api/ClosingPrice/GetClosingPriceInfo/46348559193224090"
response = requests.get(
    url,
    headers={
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Referer": "https://www.tsetmc.com/",
    },
    timeout=30,
)
response.raise_for_status()
info = response.json()["closingPriceInfo"]
print("LAST", int(info["pDrCotVal"]))
print("CLOSE", int(info["pClosing"]))
print("YDAY", int(info["priceYesterday"]))


<div class="help" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <h2 style="direction:rtl;text-align:right;">۲) load چه برمی‌گرداند</h2>
  <p><code>load("فولاد")</code> همان کد insCode را پیدا می‌کند و یک دیکشنری معمولی برمی‌گرداند. قیمت‌ها به ریال‌اند. برای هر نماد دیگری که در جستجوی سایت هست، فقط رشته را عوض کن: <code>load("فملی")</code>، <code>load("خودرو")</code>، <code>load("وبملت")</code>، <code>load("شستا")</code>.</p>
  <ul>
    <li><code>last</code> آخرین معامله تابلو، <code>close</code> قیمت پایانی، <code>yday</code> پایانی دیروز.</li>
    <li><code>high</code> و <code>low</code> سقف و کف مجاز. <code>eps</code> و <code>pe</code> سود و نسبت قیمت. <code>cap</code> ارزش بازار به ریال.</li>
    <li><code>float</code> درصد شناوری و <code>shares</code> تعداد سهم.</li>
    <li><code>history</code> لیست روزها از قدیمی به جدید. هر روز کلیدهای <code>date</code> و <code>adj</code> و <code>last</code> و <code>volume</code> را دارد. <code>adj</code> پایانی است و <code>last</code> آخرین معامله همان روز.</li>
    <li>عضو اول <code>history[0]</code>، عضو آخر <code>history[-1]</code>، بیست روز آخر <code>history[-20:]</code>.</li>
    <li><code>holders</code> لیست سهامدار عمده با کلیدهای <code>name</code> و <code>percent</code>.</li>
    <li><code>clients</code> لیست حقیقی و حقوقی، باز هم از قدیمی به جدید. عضو آخر تازه‌ترین روز است. کلید <code>real</code> خرید حقیقی و <code>legal</code> خرید حقوقی.</li>
  </ul>
  <p>داخل <code>load</code> را لازم نیست تغییر بدهی. سؤال‌ها با دیکشنری، لیست، حلقه و شرط حل می‌شوند.</p>
</div>


In [ ]:
import pytse_client as tse

# load("نام نماد") داده را از tsetmc می‌گیرد و یک دیکشنری معمولی برمی‌گرداند.
# داخل تابع، جدول کتابخانه به لیست تبدیل می‌شود. در سؤال‌ها فقط دیکشنری و لیست می‌بینی.
# بار اول هر نماد چند ثانیه طول می‌کشد. بار دوم همان نماد از حافظه خوانده می‌شود.

_cache = {}


def _day(value):
    text = str(value).split(".")[0]
    if len(text) >= 10 and text[4] == "-":
        return text[:10]
    return text


def load(symbol):
    if symbol in _cache:
        return _cache[symbol]
    ticker = tse.Ticker(symbol)
    history = []
    for row in ticker.history.to_dict("records"):
        history.append(
            {
                "date": _day(row["date"]),
                "adj": int(row["adjClose"]),
                "last": int(row["close"]),
                "volume": int(row["volume"]),
            }
        )
    holders = []
    for row in ticker.shareholders.to_dict("records"):
        holders.append(
            {
                "name": str(row["shareholder"]).strip(),
                "percent": float(row["percentage"]),
            }
        )
    clients = []
    for row in ticker.client_types.to_dict("records"):
        clients.append(
            {
                "date": _day(row["date"]),
                "real": int(float(row["individual_buy_vol"])),
                "legal": int(float(row["corporate_buy_vol"])),
            }
        )
    clients.reverse()
    data = {
        "title": ticker.title,
        "group": ticker.group_name,
        "market": ticker.flow,
        "last": int(ticker.last_price),
        "close": int(ticker.adj_close),
        "yday": int(ticker.yesterday_price),
        "high": int(ticker.sta_max),
        "low": int(ticker.sta_min),
        "eps": int(ticker.eps),
        "pe": float(ticker.p_e_ratio),
        "cap": float(ticker.market_cap),
        "float": float(ticker.float_shares),
        "shares": int(ticker.total_shares),
        "history": history,
        "holders": holders,
        "clients": clients,
    }
    _cache[symbol] = data
    return data


sample = load("فولاد")
print(sample["title"], "—", len(sample["history"]), "روز")
print('آماده است. برای نماد دیگر: load("فملی") یا load("خودرو") یا load("وبملت")')


<div class="toc" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <h2 style="direction:rtl;text-align:right;">فهرست کیس‌ها</h2>
</div>

1. [سؤال ۱ — شناسنامه نماد](#q1)
2. [سؤال ۲ — تابلوی امروز](#q2)
3. [سؤال ۳ — بازده آخرین قیمت](#q3)
4. [سؤال ۴ — دامنه مجاز](#q4)
5. [سؤال ۵ — سود و نسبت قیمت](#q5)
6. [سؤال ۶ — ارزش بازار](#q6)
7. [سؤال ۷ — طول تاریخچه](#q7)
8. [سؤال ۸ — پایانی و آخرین در تاریخچه](#q8)
9. [سؤال ۹ — بازده پنج روز](#q9)
10. [سؤال ۱۰ — سقف و کف بیست روز](#q10)
11. [سؤال ۱۱ — شلوغ‌ترین روز](#q11)
12. [سؤال ۱۲ — سیگنال بازده](#q12)
13. [سؤال ۱۳ — سهام شناور](#q13)
14. [سؤال ۱۴ — سهامدار عمده](#q14)
15. [سؤال ۱۵ — سیگنال خریدار](#q15)
16. [سؤال ۱۶ — سیگنال نهایی](#q16)


###### q1

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۱</span>
    <span class="tags">dict · str</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">شناسنامه نماد</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">با <code>load("فولاد")</code> داده فولاد را بیاور. خروجی یک دیکشنری است. سه خط چاپ کن: NAME از کلید <code>title</code>، GROUP از کلید <code>group</code>، MARKET از کلید <code>market</code>.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> هر کلید را با <code>print</code> و برچسب خودش چاپ کن. برای نماد دیگر همین تابع را با اسم آن نماد صدا بزن، مثل <code>load("فملی")</code> یا <code>load("خودرو")</code> یا <code>load("وبملت")</code>.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> کتابخانه اسم نماد را به کد insCode صفحه tsetmc وصل می‌کند. برای فولاد این کد ۴۶۳۴۸۵۵۹۱۹۳۲۲۴۰۹۰ است. تو فقط اسم نماد را می‌دهی.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">load("فولاد")
foolad["title"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">NAME فولاد مبارکه اصفهان
GROUP فلزات اساسي
MARKET بورس</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> سه خط جدا. نام‌ها همان متن سایت‌اند.</div>
</div>


In [ ]:
foolad = load("فولاد")

# To Do: نام شرکت، گروه و بازار را چاپ کن


###### q2

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۲</span>
    <span class="tags">dict · int</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">تابلوی امروز</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">تابلوی همین لحظه فولاد را چاپ کن. کلید <code>last</code> آخرین معامله است، کلید <code>close</code> قیمت پایانی است و کلید <code>yday</code> قیمت پایانی دیروز. هر سه به ریال‌اند، همان عددی که روی سایت می‌بینی. سه خط: LAST و CLOSE و YDAY.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> عددها از قبل صحیح شده‌اند. فقط چاپ کن.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> قیمت سهم در tsetmc به ریال است. ۴۴۱۰ ریال همان ۴۴۱ تومان است.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">foolad["last"]
foolad["close"]
foolad["yday"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">LAST 4410
CLOSE 4410
YDAY 4290</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> LAST آخرین معامله است و CLOSE قیمت پایانی. این دو در طول روز می‌توانند فرق داشته باشند.</div>
</div>


In [ ]:
foolad = load("فولاد")

# To Do: آخرین قیمت، قیمت پایانی و قیمت دیروز را چاپ کن


###### q3

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۳</span>
    <span class="tags">float · round</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">بازده آخرین قیمت</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">بازده آخرین قیمت فولاد نسبت به دیروز را به درصد حساب کن: (کلید last منهای کلید yday) تقسیم بر yday، ضربدر ۱۰۰. با <code>round</code> دو رقم اعشار نگه دار و با برچسب RETURN چاپ کن.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> فرمول: (last - yday) / yday * 100</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> این درصد تغییر تابلو است. مثبت یعنی آخرین معامله از پایانی دیروز بالاتر است.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">foolad["last"]
foolad["yday"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">RETURN 2.8</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> round ممکن است صفر آخر را نشان ندهد. ۲٫۸ و ۲٫۸۰ یکی‌اند.</div>
</div>


In [ ]:
foolad = load("فولاد")

# To Do: درصد تغییر آخرین قیمت نسبت به دیروز را چاپ کن


###### q4

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۴</span>
    <span class="tags">dict · int</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">دامنه مجاز</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">دامنه مجاز امروز فولاد را چاپ کن. کلید <code>high</code> سقف مجاز و کلید <code>low</code> کف مجاز است، هر دو به ریال. دو خط HIGH و LOW.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> سقف و کف مجاز را خود سایت از روی قیمت پایانی دیروز می‌سازد. خارج از این بازه سفارش ثبت نمی‌شود.</div>
  
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">foolad["high"]
foolad["low"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">HIGH 4410
LOW 4170</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> HIGH از کلید high و LOW از کلید low می‌آید.</div>
</div>


In [ ]:
foolad = load("فولاد")

# To Do: سقف و کف مجاز را چاپ کن


###### q5

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۵</span>
    <span class="tags">dict · float</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">سود و نسبت قیمت</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">سود هر سهم و نسبت قیمت به سود فولاد را چاپ کن. کلید <code>eps</code> سود هر سهم به ریال است. کلید <code>pe</code> همان P/E تابلو است. خط اول EPS. خط دوم PE با دو رقم اعشار.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> P/E قیمت را به سود هر سهم تقسیم می‌کند. هر دو عدد را سایت در صفحه نماد نشان می‌دهد.</div>
  
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">foolad["eps"]
foolad["pe"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">EPS 518
PE 8.51</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> EPS صحیح است و PE با round دو رقم اعشار می‌گیرد.</div>
</div>


In [ ]:
foolad = load("فولاد")

# To Do: سود هر سهم و P/E را چاپ کن


###### q6

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۶</span>
    <span class="tags">float · round</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">ارزش بازار</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">ارزش بازار فولاد در کلید <code>cap</code> به ریال است. آن را به میلیارد تومان تبدیل کن: اول تقسیم بر ۱۰ تا تومان شود، بعد تقسیم بر یک میلیارد. نتیجه را با round صحیح کن و با برچسب BILLION چاپ کن.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> یک میلیارد یعنی <code>1_000_000_000</code>.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> ارزش بازار برابر قیمت پایانی ضربدر تعداد سهم است. واحد سایت ریال است.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">foolad["cap"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">BILLION 853335</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> خروجی یک عدد صحیح به میلیارد تومان است.</div>
</div>


In [ ]:
foolad = load("فولاد")

# To Do: ارزش بازار را به میلیارد تومان چاپ کن


###### q7

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۷</span>
    <span class="tags">list · len</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">طول تاریخچه</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">کلید <code>history</code> لیست روزهای معاملاتی است، از قدیمی به جدید. تعداد عضو را با <code>len</code> بگیر. عضو اول <code>history[0]</code> و عضو آخر <code>history[-1]</code> است. تاریخ هر روز کلید <code>date</code> است. سه خط: DAYS و START و END.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> تاریخ از قبل به شکل YYYY-MM-DD است. ساعت ندارد.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> آخرین عضو تاریخچه ممکن است دیروز باشد. معامله امروز گاهی تا بستن فایل خروجی سایت داخل history نمی‌آید. تابلوی امروز را از کلید last بخوان.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">foolad["history"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">DAYS 4244
START 2007-03-11
END 2026-10-12</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> DAYS طول لیست است. START قدیمی‌ترین روز و END تازه‌ترین روز داخل تاریخچه است.</div>
</div>


In [ ]:
foolad = load("فولاد")
history = foolad["history"]

# To Do: تعداد روزها و تاریخ اول و آخر را چاپ کن


###### q8

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۸</span>
    <span class="tags">list · dict</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">پایانی و آخرین در تاریخچه</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">هر روز تاریخچه یک دیکشنری است. کلید <code>adj</code> قیمت پایانی آن روز است و کلید <code>last</code> آخرین معامله همان روز. این last با کلید last تابلو یکی نیست: این یکی مال آخرین روز داخل تاریخچه است. برای عضو آخر لیست، پایانی و آخرین را چاپ کن. دو خط: ADJ و LAST.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> عضو آخر لیست را با اندیس منفی یک بگیر.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> فایل خروجی قدیمی tsetmc ستون CLOSE را پایانی و LAST را آخرین معامله می‌گذارد. اینجا پایانی adj شده و آخرین معامله last.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">history[-1]["adj"]
history[-1]["last"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">ADJ 4290
LAST 4300</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> هر دو عدد مال آخرین روز موجود در تاریخچه‌اند، نه الزاما تابلوی همین دقیقه.</div>
</div>


In [ ]:
foolad = load("فولاد")
history = foolad["history"]

# To Do: پایانی و آخرینِ آخرین روز تاریخچه را چاپ کن


###### q9

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۹</span>
    <span class="tags">list · index</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">بازده پنج روز</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">بازده پنج روز معاملاتی قیمت پایانی فولاد را حساب کن. بازده برابر است با پایانی آخرین روز تقسیم بر پایانی پنج روز قبل، منهای یک، ضربدر ۱۰۰. پنج روز قبل یعنی عضو <code>history[-6]</code>، چون عضو منفی یک خود آخرین روز است. نتیجه را با دو رقم اعشار و برچسب RETURN چاپ کن. پایانی هر روز کلید <code>adj</code> است.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> اندیس منفی شش، شش عضو مانده به آخر است. از منفی یک تا منفی شش می‌شود پنج قدم عقب.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> این بازده فقط روزهای داخل لیست را می‌شمارد. پنج‌شنبه و جمعه که معامله ندارند، عضو ندارند.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">history[-1]["adj"]
history[-6]["adj"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">RETURN 13.79</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> خروجی یک درصد با دو رقم اعشار است.</div>
</div>


In [ ]:
foolad = load("فولاد")
history = foolad["history"]

# To Do: بازده پنج‌روزه قیمت پایانی را چاپ کن


###### q10

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۱۰</span>
    <span class="tags">list · for</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">سقف و کف بیست روز</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">در بیست روز آخر تاریخچه فولاد، کمترین و بیشترین قیمت پایانی را با حلقه پیدا کن. بیست روز آخر <code>history[-20:]</code> است. پایانی کلید <code>adj</code> است. دو خط LOW و HIGH.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> یک متغیر برای کف و یک متغیر برای سقف بگذار و داخل حلقه، اگر قیمتی کوچک‌تر یا بزرگ‌تر بود عوضشان کن.</div>
  
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">history[-20:]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">LOW 3210
HIGH 4290</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> LOW کف پایانی این پنجره است و HIGH سقف پایانی همان پنجره.</div>
</div>


In [ ]:
foolad = load("فولاد")
window = foolad["history"][-20:]

# To Do: کف و سقف قیمت پایانی بیست روز آخر را چاپ کن


###### q11

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۱۱</span>
    <span class="tags">list · for</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">شلوغ‌ترین روز</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">در بیست روز آخر، روزی که بیشترین حجم معامله را داشته با حلقه پیدا کن. حجم کلید <code>volume</code> است. تاریخ و حجم همان روز را در یک خط چاپ کن: اول تاریخ، بعد حجم.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> یک متغیر را روی شلوغ‌ترین روزی که تا حالا دیده‌ای نگه دار. اگر حجم روز جدید بزرگ‌تر بود، آن متغیر را عوض کن.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> حجم به عدد سهم است، نه تومان. تاریخ شلوغ، یک نشانه از توجه بازار است.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">history[-20:]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">2026-09-16 9069143306</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> خروجی یک خط است: تاریخ به شکل YYYY-MM-DD، یک فاصله، حجم صحیح.</div>
</div>


In [ ]:
foolad = load("فولاد")
window = foolad["history"][-20:]

# To Do: تاریخ و حجم شلوغ‌ترین روز این پنجره را چاپ کن


###### q12

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۱۲</span>
    <span class="tags">load · for · if</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">سیگنال بازده</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">این بار خودت سه نماد را بیاور: فولاد، فملی و خودرو. لیست <code>names</code> اسم سایت و برچسب خروجی را دارد. برای هر کدام <code>load</code> را صدا بزن و بازده آخرین قیمت نسبت به دیروز را مثل سؤال ۳ حساب کن. سه خط FOOLAD و FAMELI و KHODRO با درصد و دو رقم اعشار. خط چهارم سیگنال است: برچسب نمادی که بازده بزرگ‌تری دارد. اگر دو بازده مساوی بودند، نمادی که زودتر در لیست آمده بماند.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> یک متغیر برای بیشترین بازده و یک متغیر برای برچسبش داشته باش. فقط وقتی بازده جدید بزرگ‌تر بود هر دو را عوض کن.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> مقایسه روی درصد تغییر است، نه روی خود قیمت. قیمت ریالی دو شرکت مقیاس یکسانی برای گران‌تر نیست. خط آخر تصمیم است، نه یک عدد خام.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">load("فولاد")
load("فملی")
load("خودرو")</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">FOOLAD 2.8
FAMELI -2.38
KHODRO 2.89
KHODRO</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> چهار خط. خط آخر فقط یکی از کلمه‌های FOOLAD یا FAMELI یا KHODRO است.</div>
</div>


In [ ]:
names = [
    ("فولاد", "FOOLAD"),
    ("فملی", "FAMELI"),
    ("خودرو", "KHODRO"),
]

# To Do: برای هر نماد داده را بیاور، بازده را چاپ کن، و نماد با بازده بزرگ‌تر را انتخاب کن


###### q13

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۱۳</span>
    <span class="tags">dict · float</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">سهام شناور</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">درصد سهام شناور فولاد در کلید <code>float</code> است. تعداد کل سهم در کلید <code>shares</code> است. دو خط FLOAT و SHARES. شناوری را همان عدد درصد چاپ کن. تعداد سهم را همان عدد صحیح چاپ کن.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> شناوری درصدی از سهم است که در دسترس بازار است، نه تعداد سهم.</div>
  
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">foolad["float"]
foolad["shares"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">FLOAT 36.0
SHARES 1935000000000</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> FLOAT درصد است و SHARES تعداد سهم شرکت.</div>
</div>


In [ ]:
foolad = load("فولاد")

# To Do: درصد شناوری و تعداد سهم را چاپ کن


###### q14

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۱۴</span>
    <span class="tags">list · for</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">سهامدار عمده</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">کلید <code>holders</code> لیست سهامداران عمده است. هر عضو یک دیکشنری است: کلید <code>name</code> نام و کلید <code>percent</code> درصد مالکیت. با حلقه عضوی را پیدا کن که درصد بزرگ‌تری دارد و نام و درصد را در یک خط چاپ کن.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> مثل سؤال شلوغ‌ترین روز: یک متغیر را روی بزرگ‌ترین درصدی که تا حالا دیده‌ای نگه دار.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> این لیست را کتابخانه از صفحه سهامداران همان نماد می‌خواند. درصدها مال دارندگان عمده‌اند و جمعشان صد نمی‌شود.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">foolad["holders"]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">سازمان توسعه ونوسازي معادن وصنايع معدني ايران 16.709</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> یک خط: نام سهامدار، یک فاصله، درصد.</div>
</div>


In [ ]:
foolad = load("فولاد")
holders = foolad["holders"]

# To Do: نام و درصد بزرگ‌ترین سهامدار را چاپ کن


###### q15

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۱۵</span>
    <span class="tags">list · if</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">سیگنال خریدار</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">کلید <code>clients</code> خرید حقیقی و حقوقی است و مثل تاریخچه از قدیمی به جدید چیده شده. عضو آخر، تازه‌ترین روز است. کلید <code>real</code> حجم خرید حقیقی است و کلید <code>legal</code> حجم خرید حقوقی. سه خط چاپ کن: DATE تاریخ، REAL حجم حقیقی، LEGAL حجم حقوقی. خط چهارم سیگنال است. اگر حجم حقیقی بیشتر یا مساوی حقوقی بود REAL را چاپ کن، وگرنه LEGAL.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> حقیقی شخص است و حقوقی شرکت. خط آخر می‌گوید آن روز کدام سمت بیشتر خریده است.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> تاریخ این لیست به شکل عددی YYYYMMDD است. حجم‌ها عدد سهم‌اند، نه ارزش ریالی.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">foolad["clients"][-1]</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">DATE 20261012
REAL 902675077
LEGAL 284155256
REAL</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> خط چهارم فقط یکی از دو کلمه REAL یا LEGAL است. این تصمیم از مقایسه دو حجم درمی‌آید.</div>
</div>


In [ ]:
foolad = load("فولاد")
latest = foolad["clients"][-1]

# To Do: تاریخ و حجم خرید حقیقی و حقوقی را چاپ کن
# خط آخر سیگنال است: REAL یا LEGAL


<div class="guide" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <h2 style="direction:rtl;text-align:right;">از عدد به سیگنال</h2>
  <p>سؤال‌های قبل عدد را از سایت درآوردند: بازده یک روز، بازده پنج روز، شلوغ‌ترین روز، و حجم حقیقی در برابر حقوقی. هیچ‌کدام به‌تنهایی تصمیم نیستند. سیگنال قاعده‌ای است که چند عدد را به یک کلمه تبدیل می‌کند.</p>
  <p>قاعده سؤال آخر دو شرط دارد. هر دو را خودت روی داده زنده حساب می‌کنی.</p>
  <ol>
    <li><b>قیمت:</b> بازده آخرین معامله نسبت به دیروز مثبت باشد. فرمول همان سؤال ۳ است.</li>
    <li><b>خریدار:</b> در تازه‌ترین روز، حجم خرید حقیقی از حجم خرید حقوقی بیشتر باشد. همان دو عددی که در سؤال ۱۵ دیدی.</li>
  </ol>
  <ul>
    <li>هر دو شرط برقرار باشد: <b>BUY</b></li>
    <li>فقط یکی برقرار باشد: <b>WATCH</b></li>
    <li>هیچ‌کدام برقرار نباشد: <b>WAIT</b></li>
  </ul>
  <p>لیست همان سه نماد سؤال ۱۲ است. جواب‌ها لازم نیست یکی باشند. اگر یکی BUY شد و دیگری WATCH، قاعده دارد تفاوت داده را نشان می‌دهد.</p>
  <div class="box warn" dir="rtl"><b>معنی کلمه:</b> BUY این دفترچه توصیه خرید نیست. فقط اسم قاعده‌ای است که خودت نوشتی.</div>
</div>


###### q16

<div class="qcard" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <div class="qbar" dir="rtl" style="direction:rtl;text-align:right;">
    <span class="qnum">سؤال ۱۶</span>
    <span class="tags">if · and</span>
  </div>
  <h2 style="direction:rtl;text-align:right;">سیگنال نهایی</h2>
  <div class="qbody" dir="rtl" style="direction:rtl;text-align:right;">همان سه نماد را با یک قاعده دوشرطی علامت بزن. شرط اول: بازده آخرین قیمت نسبت به دیروز مثبت باشد. شرط دوم: حجم خرید حقیقی تازه‌ترین روز از حجم خرید حقوقی بیشتر باشد. اگر هر دو برقرار بود BUY، اگر فقط یکی برقرار بود WATCH، اگر هیچ‌کدام نبود WAIT. سه خط چاپ کن. هر خط برچسب نماد و یکی از این سه کلمه است.</div>
  <div class="box tip" dir="rtl"><b>راهنما:</b> بازده را مثل سؤال ۳ حساب کن. حقیقی و حقوقی، عضو آخر کلید clients است. هر دو شرط را با <code>and</code> بنویس و یکی از دو شرط را با <code>or</code>.</div>
  <div class="box extra" dir="rtl"><b>راهنمای مفهوم کمکی:</b> بازده صفر شرط اول را برقرار نمی‌کند. برابر بودن دو حجم هم شرط دوم را برقرار نمی‌کند.</div>
  <div class="box warn" dir="rtl"><b>عدد زنده:</b> عددهای قالب خروجی مال یک لحظه بازارند. کد درست، قیمت همین حالا را چاپ می‌کند و لازم نیست رقم‌به‌رقم با قالب یکی باشد. شکل خط‌ها باید یکی باشد.</div>
  <div class="io">
    <div class="pane in">
      <div class="pane-label">از کجا</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">load("فولاد")
load("فملی")
load("خودرو")</pre>
    </div>
    <div class="pane out">
      <div class="pane-label">قالب خروجی</div>
      <pre dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate;white-space:pre;">FOOLAD BUY
FAMELI WATCH
KHODRO BUY</pre>
    </div>
  </div>
  <div class="box why" dir="rtl"><b>چرا این شکل؟</b> سه خط. کلمه آخر هر خط فقط BUY یا WATCH یا WAIT است. این کلمه از جمع دو شرط درمی‌آید، نه از یک عدد تنها.</div>
</div>


In [ ]:
names = [
    ("فولاد", "FOOLAD"),
    ("فملی", "FAMELI"),
    ("خودرو", "KHODRO"),
]

# To Do: برای هر نماد دو شرط را حساب کن و BUY یا WATCH یا WAIT را چاپ کن


<div class="guide" dir="rtl" align="right" style="direction:rtl;text-align:right;">
  <h2 style="direction:rtl;text-align:right;">قاعده را عوض کن</h2>
  <p>سؤال ۱۶ تمام شدنی است: سه نماد، دو شرط، یک کلمه برای هر کدام. مسیر کار همین بود که دیدی. داده را با <code>load</code> بیاور، عددها را جدا حساب کن، بعد با <code>if</code> به یک کلمه برسان.</p>
  <p>قاعده را می‌توانی عوض کنی، بی‌آنکه مسیر عوض شود. نماد تازه‌ای را به لیست اضافه کن. یا به‌جای بازده یک‌روزه، بازده پنج‌روزه سؤال ۹ را شرط اول بگذار. حلقه همان حلقه است و کلمه خروجی مال قاعده جدید توست.</p>
</div>
